# IRS Reference PDF Pre-Ingestion

This notebook ingests publicly available IRS reference PDFs (p17, 1040, 1099, etc.)
into ChromaDB so they are available for RAG queries in the tax-advisor CLI.

**PII redaction is skipped** because these are public government documents — redaction
would corrupt legitimate dates, names, and numbers in tax rules.

In [2]:
"""Cell 1 — Setup: imports, settings, docs directory."""

from pathlib import Path

from rich.console import Console

from tax_advisor.config import Settings
from tax_advisor.ingestion.loader import discover_pdfs
from tax_advisor.rag.pipeline import ingest_documents, get_index_stats, query_documents

# --- Configuration ---
# Point this at the local folder containing IRS PDFs (p17.pdf, i1040.pdf, etc.)
DOCS_DIR = Path("../documents")
TAX_YEAR = "2025"

settings = Settings()
console = Console()

print(f"Docs directory : {DOCS_DIR.resolve()}")
print(f"Chroma dir     : {settings.chroma_dir}")
print(f"Collection     : {settings.chroma_collection}")
print(f"Embedding model: {settings.embedding_model}")
print(f"Tax year       : {TAX_YEAR}")

Docs directory : /Users/kntc847/code/my-personal/tax-advisor/documents
Chroma dir     : chroma_db
Collection     : tax_documents
Embedding model: amazon.titan-embed-text-v2:0
Tax year       : 2025


In [3]:
"""Cell 2 — Discover: list PDFs found in the folder."""

pdfs = discover_pdfs(DOCS_DIR)

if not pdfs:
    print(f"⚠ No PDFs found in {DOCS_DIR.resolve()}")
    print("  Place IRS PDFs (p17.pdf, i1040.pdf, etc.) in the folder and re-run.")
else:
    print(f"Found {len(pdfs)} PDF(s):")
    for p in sorted(pdfs):
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"  • {p.name}  ({size_mb:.1f} MB)")

Found 2 PDF(s):
  • f1040.pdf  (0.2 MB)
  • p17.pdf  (2.9 MB)


In [ ]:
"""Cell 3 — Ingest: convert, chunk, embed, and index (no PII redaction)."""

count = ingest_documents(
    DOCS_DIR,
    settings,
    console,
    skip_redaction=True,
    tax_year=TAX_YEAR,
)

print(f"\nIngested {count} document(s).")

Found 2 PDF(s) in ../documents

Processing f1040.pdf …

→ 7 chunk(s)

✓ indexed

Processing p17.pdf …

In [ ]:
"""Cell 4 — Verify: show index stats and run a sample search."""

stats = get_index_stats(settings)
print("Index stats:")
print(f"  Collection    : {stats['collection']}")
print(f"  Document count: {stats['document_count']}")

# Sample query
SAMPLE_QUERY = "What is the standard deduction for 2025?"
print(f"\n--- Sample query: {SAMPLE_QUERY!r} ---\n")

result = query_documents(SAMPLE_QUERY, settings, n_results=3)
print(result)